
**Created by:**

__Viktor Varga__

<br>
<img src="../_files/nb_im/ik_logo.png" style="display:inline-block" width='40%'>
<hr>

# Python tutorial - NumPy: Advanced `ndarray` shape manipulation, and the details of how views work

## `ndarray.strides`

When we create a NumPy array (for example with `np.array()`, `np.zeros()`, `np.ones()`, `np.random()`, etc.), a contiguous region is allocated in memory. Regardless of how many dimensions the array appears to have, memory addressing is sequential, not multidimensional.

The `ndarray` implementation lets us interpret this memory region (or a part of it) as if it were a multidimensional array. The implementation hides the conversion between multidimensional coordinates and memory addresses, so we usually do not have to deal with it directly. Still, by clarifying the behavior of the `ndarray.shape` and `ndarray.strides` attributes, it becomes much easier to understand how arrays and views work.


Let's create a 2D array of type `np.int32`! Check the array shape (`shape`), its strides (`strides`), and the per-element size in memory (`itemsize`).

In [ ]:
import numpy as np

a = np.arange(6, dtype=np.int32).reshape((2,3))

print("The array 'a':\n", a)
print("  ... its shape:", a.shape)
print("  ... its strides:", a.strides)
print("  ... size of its items (in byte):", a.itemsize)

The array 'a':
 [[0 1 2]
 [3 4 5]]
  ... its shape: (2, 3)
  ... its strides: (12, 4)
  ... size of its items (in byte): 4


The array `a` has shape `(2, 3)`, strides `(12, 4)`, and each element occupies 4 bytes.

The `strides` attribute tells you how many **bytes** you have to move in memory to increase the index by 1 along each axis. For example, starting from `a[0, 0]`, we have to move 4 bytes to reach `a[0, 1]`, we have to move 12 byte to reach `a[1, 0]`, and to reach `a[1, 1]` we have to move 12 + 4 = 16 bytes. If we divide these values by the element size (`int32` = 32 bit = 4 bytes), we get that a single step along axis #0 corresponds to 3 elements in memory, while a single step along axis #1 corresponds to 1 element in memory. The figure below illustrates this:

<img src='../_files/nb_im/numpy_strides1-strides1.png' width='60%'>

In general, relative to `a[0, 0, 0, ...]`, to reach `a[idx0, idx1, idx2, ...]` you need to move `(idx0*a.strides[0] + idx1*a.strides[1] + idx2*a.strides[2] + ...)` bytes in memory.


## Changing the array shape

Let us see how changing the shape of a 12-element `int32` array affects the strides!


In [ ]:
a = np.arange(12, dtype=np.int32).reshape((4,3))

print("The 2D array 'a':\n", a)
print("  ... its shape:", a.shape)
print("  ... its strides:", a.strides)

a = a.reshape((-1))
print("\nThe flattened array 'a':\n", a)
print("  ... its shape:", a.shape)
print("  ... its strides:", a.strides)

a = a.reshape((2,2,3))
print("\nThe 3D array 'a':\n", a)
print("  ... its shape:", a.shape)
print("  ... its strides:", a.strides)




The 2D array 'a':
 [[ 0  1  2]
 [ 3  4  5]
 [ 6  7  8]
 [ 9 10 11]]
  ... its shape: (4, 3)
  ... its strides: (12, 4)

The flattened array 'a':
 [ 0  1  2  3  4  5  6  7  8  9 10 11]
  ... its shape: (12,)
  ... its strides: (4,)

The 3D array 'a':
 [[[ 0  1  2]
  [ 3  4  5]]

 [[ 6  7  8]
  [ 9 10 11]]]
  ... its shape: (2, 2, 3)
  ... its strides: (24, 12, 4)


Nothing unexpected happens: the flattened, 1D version of the array has strides `(4,)`, meaning that along its only axis we have to step 4 bytes (that is, one element) in memory to increment the index.

The strides of the two-dimensional array with shape (4, 3) are: (12, 4), since increasing the index along the first axis by one skips 3 elements (the length of the second axis) in memory.

If we reshape the 12-element array into a 3D array with shape `(2, 2, 3)`, the stride along the first axis becomes larger: 24 bytes, that is, 6 elements: the product of the lengths of all the axes following the first one.


## Basic indexing

The idea of *basic indexing* is to reference multiple elements of an array at once using regular (constant) step sizes. If we create a new array via basic indexing from another array, the result is still an `ndarray` (just like the original), so at first glance a view and its _base array_ are indistinguishable. However, the memory region referenced by a view is not necessarily contiguous, and it does not necessarily cover the full memory region originally allocated for the array.

Let us see how the strides change if we take a view of a single column from a 2D array.


In [ ]:
a = np.arange(6, dtype=np.int32).reshape((2,3))
print("The 2D array 'a':\n", a)
print("  ... its shape:", a.shape)
print("  ... its strides:", a.strides)

b = a[:,1]
print("\nThe view 'b' of array 'a':\n", b)
print("  ... its shape:", b.shape)
print("  ... its strides:", b.strides)

The 2D array 'a':
 [[0 1 2]
 [3 4 5]]
  ... its shape: (2, 3)
  ... its strides: (12, 4)

The view 'b' of array 'a':
 [1 4]
  ... its shape: (2,)
  ... its strides: (12,)


As we can see, even though elements in the view `b` are still 4 bytes each, iterating over `b` requires jumping by 12 bytes as we do not reference all the elements anymore from the original array. In other words, the view points into a slice of the original memory region, but with modified `shape` and `strides`, which changes how iteration proceeds.

Let us look at a slightly more complex example of basic indexing.


In [ ]:
a = np.arange(6, dtype=np.int32).reshape((2,3))
print("The 2D array 'a':\n", a)
print("  ... its shape:", a.shape)
print("  ... its strides:", a.strides)

b = a[1::-2]
print("\nThe view 'b':\n", b)
print("  ... its shape:", b.shape)
print("  ... its strides:", b.strides)

The 2D array 'a':
 [[0 1 2]
 [3 4 5]]
  ... its shape: (2, 3)
  ... its strides: (12, 4)

The view 'b':
 [[3 4 5]]
  ... its shape: (1, 3)
  ... its strides: (-24, 4)


When indexing with a negative step, the corresponding entry in `strides` becomes negative.

This is rarely needed in practice, but you can distinguish a base array (one that originally allocated the referenced memory) from a view by inspecting `ndarray.base`. For a view, this attribute points to the "original array", while for a base array it is `None`.


In [ ]:
a = np.arange(6, dtype=np.int32)
print("The base of the original 'a' array:", a.base)  # None, as 'a' is an original array

b = a.reshape((2,3))
print("The base of the reshaped 'b' view:", b.base)   # not None, since 'b' is a view

c = b[:,1]
print("The base of the view 'c':", c.base)   # not None, since 'c' is a view

The base of the original 'a' array: None
The base of the reshaped 'b' view: [0 1 2 3 4 5]
The base of the view 'c': [0 1 2 3 4 5]


From the examples above, it should be clear why basic indexing produces a view, while advanced indexing requires a copy. With basic indexing, iteration over the referenced memory region can be described with regular step sizes and the lengths of the iteration over the axes, so there is no need to create a copy of the underlying data, but only a new `ndarray` object over the same memory region with different shape and strides. With advanced indexing, arbitrary elements can be selected, so the referenced locations usually cannot be described with regular step sizes, therefore a new array allocation is necessary to extract the indexed items.


## Swapping axes, transposition

An array's axes can be swapped easily, which can also be done via a view, without allocating a new array (most of the time). One frequent use case is transposing matrices. Let us see how the strides change under transposition.


In [ ]:
a = np.arange(6, dtype=np.int32).reshape((2,3))
print("The 2D array 'a':\n", a)
print("  ... its shape:", a.shape)
print("  ... its strides:", a.strides)

b = np.transpose(a)
print("\nThe transposed array:\n", b)
print("  ... its shape:", b.shape)
print("  ... its strides:", b.strides)

The 2D array 'a':
 [[0 1 2]
 [3 4 5]]
  ... its shape: (2, 3)
  ... its strides: (12, 4)

The transposed array:
 [[0 3]
 [1 4]
 [2 5]]
  ... its shape: (3, 2)
  ... its strides: (4, 12)


As we can see, transposition swapped both the axis lengths and the strides required to increment indices.

## Broadcasting

Broadcasting is the process of making an array's shape compatible with another array's shape. This happens, for example, when we want to perform an elementwise operation on two arrays and need to align their elements, which requires matching shapes.

<img src="../_files/nb_im/numpy_broadcast1-br2.png" style="display:inline-block" width='50%'>

The figure above shows an example of how shape alignment works during broadcasting. The second array originally has shape `(3,)`: we first create a new axis, then repeat the array along that new axis to reach shape `(2, 3)`.

However, the repetition along the new axis is only *virtual*, as the `strides` attribute of a broadcasted array reveals. Since broadcasting usually happens automatically when calling an appropriate elementwise operation, to inspect the view created by broadcasting we use the explicit `np.broadcast_to()` operation, which performs the shape alignment without applying any additional operation.


In [ ]:
a = np.arange(6, dtype=np.int32).reshape((2,3))
print("The 2D array 'a':\n", a)
print("  ... its shape:", a.shape)
print("  ... its strides:", a.strides)

b = np.arange(3, dtype=np.int32)
print("\nThe 1D array 'b':\n", b)
print("  ... its shape:", b.shape)
print("  ... its strides:", b.strides)

# We will broadcast 'b' to the shape of 'a'

b = np.broadcast_to(b, a.shape)
print("\nThe 'b' array broadcast to the shape of 'a':\n", b)
print("  ... its shape:", b.shape)
print("  ... its strides:", b.strides)

The 2D array 'a':
 [[0 1 2]
 [3 4 5]]
  ... its shape: (2, 3)
  ... its strides: (12, 4)

The 1D array 'b':
 [0 1 2]
  ... its shape: (3,)
  ... its strides: (4,)

The 'b' array broadcast to the shape of 'a':
 [[0 1 2]
 [0 1 2]]
  ... its shape: (2, 3)
  ... its strides: (0, 4)


Notice that the array `b` broadcasted to shape `(2, 3)` has a stride of size 0 along the new axis (axis #0). This simply means that even though the apparent shape is `(2, 3)`, the underlying data is only `(1, 3)`, and indexing along axis #0 has no effect: it will read the same elements regardless of the value of that index. Therefore it is clear, that the repetition of the data during broadcasting is purely virtual, and thus much more efficient than achieving the same functionality by explicit tiling of the data to match shapes.


## Stride tricks

By manually adjusting strides, we can obtain custom behavior that makes certain operations easy to implement.

One example is *windowed* processing. Suppose we have a long 1D array and we want to compute averages over overlapping windows. This is a common task in signal processing.


In [ ]:
a = np.arange(7, dtype=np.float32)
print("The 1D array 'a':\n", a)
print("  ... its shape:", a.shape)
print("  ... its strides:", a.strides)

window_len = 4

idxs = np.stack([np.arange(i, i+window_len) for i in range(a.shape[0]-window_len+1)], axis=0)
windowed_a = a[idxs]
print("\nThe windowed 'a' array:\n", windowed_a)
print("  ... its shape:", windowed_a.shape)
print("  ... its strides:", windowed_a.strides)

window_means = np.mean(windowed_a, axis=1)
print("\nThe windowed mean values:\n", window_means)

The 1D array 'a':
 [0. 1. 2. 3. 4. 5. 6.]
  ... its shape: (7,)
  ... its strides: (4,)

The windowed 'a' array:
 [[0. 1. 2. 3.]
 [1. 2. 3. 4.]
 [2. 3. 4. 5.]
 [3. 4. 5. 6.]]
  ... its shape: (4, 4)
  ... its strides: (16, 4)

The windowed mean values:
 [1.5 2.5 3.5 4.5]


The above is a naive solution for creating windows and averaging them. To enumerate the windows, we need an array of approximately `window_size * sequence_length` elements, which can consume a lot of memory for large window sizes and long sequences. However, most of the elements in this windowed array are redundant, because the same original elements are repeated many times.

A better idea is that indexing along different axes does not have to cover disjoint memory regions: overlaps are allowed. Next to the original `sequence_length` axis we create a new axis that indexes the elements *within* each window.

To specify strides manually, we will use `numpy.lib.stride_tricks.as_strided()`.

**Important: incorrect use of this function can lead to segmentation faults, because with an invalid shape/stride combination you can index outside the array (or even outside memory owned by the Python process).**

By default, the function returns a read-only view of the original array. If we set the `writeable` parameter to `True`, the view becomes writable. Do not do this: you could overwrite the memory of other Python objects. Moreover, even with "correct" usage, writing into a view composed of overlapping slices has unpredictable results.


In [ ]:
a = np.arange(7, dtype=np.float32)
print("The 1D array 'a':\n", a)
print("  ... its shape:", a.shape)
print("  ... its strides:", a.strides)

window_len = 4

new_shape = (a.shape[0]-window_len+1, window_len)
new_strides = (a.itemsize, a.itemsize)

windowed_a = np.lib.stride_tricks.as_strided(a, shape=new_shape, strides=new_strides)
print("\nThe windowed 'a' array:\n", windowed_a)
print("  ... its shape:", windowed_a.shape)
print("  ... its strides:", windowed_a.strides)

window_means = np.mean(windowed_a, axis=1)
print("\nThe windowed mean values:\n", window_means)

The 1D array 'a':
 [0. 1. 2. 3. 4. 5. 6.]
  ... its shape: (7,)
  ... its strides: (4,)

The windowed 'a' array:
 [[0. 1. 2. 3.]
 [1. 2. 3. 4.]
 [2. 3. 4. 5.]
 [3. 4. 5. 6.]]
  ... its shape: (4, 4)
  ... its strides: (4, 4)

The windowed mean values:
 [1.5 2.5 3.5 4.5]


With `as_strided()`, we computed windowed averages without allocating a new array to physically hold the data structured in the form of windows. We set the view's strides so that increasing the index by one will move by exactly one element along both axes. This is the correct behavior as we need to step by one element both when moving between windows and when moving within a window in the original array.

The memory requirement drops to a fraction of the previous approach.